# agent-friend — Personal AI Agent Demo

A personal AI agent with memory, web search, code execution, file access, voice, scheduled tasks, SQLite databases, and custom tools. One pip install.

**Free to use** — get a free API key at [openrouter.ai](https://openrouter.ai/) (no credit card required).

| | |
|---|---|
| 📦 GitHub | [github.com/0-co/agent-friend](https://github.com/0-co/agent-friend) |
| 🧪 Tests | 423 passing |
| 🔑 Free tier | OpenRouter — Gemini 2.0 Flash, Llama 3.3 70B |

---

## Setup
1. Get a free API key at [openrouter.ai](https://openrouter.ai/)
2. Run the **Install** cell
3. Enter your key in the **API Key** cell
4. Run any demo below


In [ ]:
# Install agent-friend (takes ~30 seconds)
!pip install "git+https://github.com/0-co/agent-friend.git[all]" -q
print("✓ agent-friend installed")

In [ ]:
import os
import getpass

# Option A: OpenRouter (FREE — no credit card, just an account)
# Get your key at https://openrouter.ai/
print("Get a free key at https://openrouter.ai/")
key = getpass.getpass("Enter your OpenRouter API key (sk-or-...): ")
if key:
    os.environ["OPENROUTER_API_KEY"] = key
    print("✓ OpenRouter key set")

# Option B: Anthropic
# key = getpass.getpass("Anthropic API key (sk-ant-...): ")
# os.environ["ANTHROPIC_API_KEY"] = key

# Option C: OpenAI
# key = getpass.getpass("OpenAI API key (sk-...): ")
# os.environ["OPENAI_API_KEY"] = key

## Demo 1 — Basic Chat

`Friend` is the main class. Give it a seed (system prompt) and start chatting.
Using OpenRouter's free Gemini 2.0 Flash model here.

In [ ]:
from agent_friend import Friend

friend = Friend(
    seed="You are a concise, helpful assistant.",
    model="google/gemini-2.0-flash-exp:free",  # free tier
)

response = friend.chat("What is an AI agent and why do they need special infrastructure?")
print(response.text)

## Demo 2 — Web Search

Add `"search"` to the tools list. The agent can search the web and incorporate results.

In [ ]:
researcher = Friend(
    seed="You are a research assistant. Be concise and cite sources.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["search"],
)

response = researcher.chat("What are the most popular open-source AI agent frameworks in 2026?")
print(response.text)

## Demo 3 — Memory Across Turns

Add `"memory"` to persist information across conversations. Memory is stored as a local JSON file
and loaded automatically on each session.

In [ ]:
assistant = Friend(
    seed="You are a personal assistant. Remember important things about the user.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["memory"],
)

# Store some information
assistant.chat("My name is Alice and I'm building an AI agent for customer support.")
assistant.chat("I prefer concise responses and I use Python.")

# Test recall — starts fresh conversation but memory persists
assistant.reset()  # clears conversation history but not memory store
response = assistant.chat("What do you know about me?")
print(response.text)

## Demo 4 — Code Execution

Add `"code"` to let the agent write and execute Python code in a sandbox.

In [ ]:
coder = Friend(
    seed="You are a coding assistant. When asked to compute, write and run Python code.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["code"],
)

response = coder.chat(
    "Find the first 10 prime numbers greater than 100 and calculate their sum. Show your work."
)
print(response.text)

## Demo 5 — All Tools Together

Combine search, fetch, memory, code, and file access for a full-capability agent.

In [ ]:
full_agent = Friend(
    seed="""You are a capable personal agent. 
    You can search the web, fetch URLs, remember things, write and run code, and read/write files.
    Be direct and practical.""",
    model="google/gemini-2.0-flash-exp:free",
    tools=["search", "fetch", "memory", "code", "file"],
    budget_usd=0.10,  # spend limit — raises BudgetExceeded if exceeded
)

response = full_agent.chat(
    "Search for the latest Python release, then fetch https://www.python.org/downloads/ "
    "and tell me the current stable version."
)
print(response.text)

## Demo 6 — Fetch a URL

Fetch any URL and read its content. HTML is auto-converted to plain text. No API key needed.

In [ ]:
reader = Friend(
    seed="You summarize web content clearly and concisely.",
    model="google/gemini-2.0-flash-exp:free",
    tools=["fetch"],
)

# Fetch a real URL and summarize it
response = reader.chat("Fetch https://pypi.org/pypi/requests/json and tell me the latest version of requests and its release date.")
print(response.text)

## Demo 7 — CLI

Run agent-friend from the command line.

In [ ]:
# Interactive mode (requires terminal — run this locally, not in Colab)
# agent-friend -i --tools search,memory,code,file

# One-shot mode:
!agent-friend --model google/gemini-2.0-flash-exp:free "What time is it right now?"

## Demo 8 — Voice

Add `"voice"` to let your agent speak responses aloud. Uses system TTS (espeak on Linux, `say` on macOS, PowerShell on Windows) — no API key needed. For neural voices, point it at any HTTP TTS server.

Run this locally to hear the agent speak. In Colab, set `AGENT_FRIEND_TTS_URL` to an HTTP TTS server URL.

In [ ]:
import os
from agent_friend import Friend, VoiceTool

# VoiceTool: no API key needed for system TTS
# - Local: uses espeak (Linux), say (macOS), PowerShell (Windows) automatically
# - HTTP/Colab: set AGENT_FRIEND_TTS_URL to a TTS server for neural voices
tts_url = os.environ.get("AGENT_FRIEND_TTS_URL")

speaker = Friend(
    seed="You are a concise assistant. When asked to speak, use the voice tool.",
    model="google/gemini-2.0-flash-exp:free",
    tools=[VoiceTool(tts_url=tts_url)],
    budget_usd=0.02,
)

response = speaker.chat(
    "Use the voice tool to say: 'agent-friend can search, remember, run code, and speak.' "
    "Then tell me what happened."
)
print(response.text)
# Run locally to hear it — espeak (Linux), say (macOS), or PowerShell (Windows) required

## Demo 9 — RSS Feeds

Add `"rss"` to subscribe to RSS and Atom feeds. No API key required — just a URL.

`read_feed(name)` returns the latest items from a subscribed feed.

In [ ]:
from agent_friend import Friend

# RSSFeedTool: no API key needed — just a URL
# fetch_feed works directly; subscribe + read_feed for named shortcuts
rss_agent = Friend(
    seed="You are a helpful news summarizer.",
    tools=["rss"],
    api_key=OPENROUTER_API_KEY,
    model="google/gemini-2.0-flash-exp:free",
)

# Subscribe to HN and read the top 3 stories
response = rss_agent.chat(
    "Subscribe the Hacker News RSS at https://news.ycombinator.com/rss with name hn, "
    "then read the top 3 stories and give me a one-sentence summary of each."
)
print(response.text)

## Demo 10 — Scheduled Tasks

Add `"scheduler"` to schedule tasks for your agent to run on a timer or at a specific time.

`run_pending()` returns tasks that are due — combine with a cron job or `agent-friend schedule` CLI.

In [ ]:
from agent_friend import Friend, SchedulerTool
import datetime

# SchedulerTool: zero dependencies, stores in ~/.agent_friend/scheduler.json
scheduler_agent = Friend(
    seed="You manage scheduled tasks. Be concise about what you've scheduled.",
    tools=["scheduler"],
    api_key=OPENROUTER_API_KEY,
    model="google/gemini-2.0-flash-exp:free",
)

# Schedule a recurring task (runs every 1440 minutes = daily)
response = scheduler_agent.chat(
    "Schedule a task called 'daily_news' with prompt 'Search for AI agent news and summarize top 3' "
    "to run every 1440 minutes. Then list all scheduled tasks."
)
print(response.text)

# Directly inspect pending tasks
tool = SchedulerTool()
tasks = tool.list_scheduled()
print(f"\nScheduled tasks: {len(tasks)}")
for t in tasks:
    print(f"  - {t['id']}: runs every {t.get('interval_minutes')} min, next: {t['next_run']}")

## Demo 11 — SQLite Database

Add `"database"` to give your agent a real relational database. It can create tables, insert rows, run queries, and inspect schemas — all backed by a local SQLite file. Zero dependencies.

In [ ]:
from agent_friend import Friend, DatabaseTool

# DatabaseTool: no API key needed for the Python API
db = DatabaseTool()
db.create_table("tasks", "id INTEGER PRIMARY KEY, title TEXT, done INTEGER DEFAULT 0")
db.insert("tasks", {"title": "Ship v0.9", "done": 0})
db.insert("tasks", {"title": "Write tests", "done": 1})
db.insert("tasks", {"title": "Update Colab", "done": 1})

print(db.query("SELECT * FROM tasks"))
print(db.query("SELECT COUNT(*) as total FROM tasks WHERE done = 1"))

# With the full agent:
# agent = Friend(tools=["database"])
# agent.chat("Create a notes table and add three ideas")
# agent.chat("Show me all my notes")


## Demo 12 — Custom Tools with `@tool`

Register any Python function as an agent tool. Type hints become the JSON schema automatically.
The decorated function remains callable normally.


In [ ]:
from agent_friend import Friend, tool

# @tool turns any function into an agent tool
# Type hints → JSON schema. Optional params are not required.

@tool
def stock_price(ticker: str) -> str:
    """Get current stock price for a ticker symbol."""
    prices = {"AAPL": "182.50", "GOOG": "175.20", "NVDA": "875.00"}
    return f"{ticker.upper()}: ${prices.get(ticker.upper(), '???')}"

@tool(name="celsius_to_fahrenheit", description="Convert Celsius to Fahrenheit")
def to_fahrenheit(celsius: float) -> str:
    return f"{celsius:.1f}°C = {celsius * 9/5 + 32:.1f}°F"

# Functions still work normally
print(stock_price("AAPL"))    # AAPL: $182.50
print(to_fahrenheit(22.0))    # 22.0°C = 71.6°F

# Mix custom tools with built-in tools
agent = Friend(
    seed="You are a helpful assistant.",
    tools=["search", stock_price, to_fahrenheit],
)
response = agent.chat(
    "What's AAPL stock price? Also convert 25°C to Fahrenheit."
)
print(response.text)


---

## What's next?

- **Full docs**: [github.com/0-co/agent-friend](https://github.com/0-co/agent-friend)
- **More agent-* tools**: [github.com/0-co/company](https://github.com/0-co/company) — 21 packages for reliability, observability, testing, safety, and state
- **Watch it being built**: [twitch.tv/0coceo](https://twitch.tv/0coceo) — an AI building a company, live
- **Custom tools**: use `@tool` to register any function — stock prices, internal APIs, database queries
- **File a bug or request a feature**: [github.com/0-co/agent-friend/issues](https://github.com/0-co/agent-friend/issues)
